In [3]:
import requests
import csv
from datetime import datetime, timedelta

# --- CONFIGURATION ---
API_KEY = "NbQjemluM7n4CNWIlMO4oe0M95C2APUp9FPtgiTO"
DAYS_TO_FETCH = 30
FILE_NAME = "asteroid_data.csv"

def fetch_asteroids():
    end_date = datetime.now()
    start_date = end_date - timedelta(days=DAYS_TO_FETCH)

    current_start = start_date
    all_data = []

    print(f"Fetching data for the last {DAYS_TO_FETCH} days...")

    while current_start < end_date:
        # API only allows 7-day chunks
        current_end = min(current_start + timedelta(days=6), end_date)

        # Build URL carefully
        base_url = "https://api.nasa.gov/neo/rest/v1/feed"
        params = {
            'start_date': current_start.strftime('%Y-%m-%d'),
            'end_date': current_end.strftime('%Y-%m-%d'),
            'api_key': API_KEY
        }

        response = requests.get(base_url, params=params)

        if response.status_code != 200:
            print(f"Error fetching data: {response.status_code}")
            break

        data = response.json()
        daily_objects = data.get('near_earth_objects', {})

        for date in daily_objects:
            for asteroid in daily_objects[date]:
                # Extracting specific fields for SQL
                all_data.append({
                    'id': asteroid['id'],
                    'name': asteroid['name'],
                    'magnitude': asteroid['absolute_magnitude_h'],
                    'diameter_min_km': asteroid['estimated_diameter']['kilometers']['estimated_diameter_min'],
                    'diameter_max_km': asteroid['estimated_diameter']['kilometers']['estimated_diameter_max'],
                    'velocity_kph': asteroid['close_approach_data'][0]['relative_velocity']['kilometers_per_hour'],
                    'miss_distance_km': asteroid['close_approach_data'][0]['miss_distance']['kilometers'],
                    'is_hazardous': 1 if asteroid['is_potentially_hazardous_asteroid'] else 0
                })

        # Move to the next chunk
        current_start = current_end + timedelta(days=1)

    if all_data:
        # Save to CSV
        keys = all_data[0].keys()
        with open(FILE_NAME, 'w', newline='', encoding='utf-8') as output_file:
            dict_writer = csv.DictWriter(output_file, fieldnames=keys)
            dict_writer.writeheader()
            dict_writer.writerows(all_data)
        print(f"Successfully saved {len(all_data)} asteroids to {FILE_NAME}")
    else:
        print("No data found.")

if __name__ == "__main__":
    fetch_asteroids()

Fetching data for the last 30 days...
Successfully saved 128 asteroids to asteroid_data.csv
